# Phase A: Project Scaffolding – Offline Smoke Test

This notebook verifies the Phase A public API **without touching hardware,**
**COM ports, Ethernet, DCA, or any real project directories.**

All work is done inside a `tempfile.TemporaryDirectory`.

In [1]:
# Cell 1 – Canonical package import
import sys, os, tempfile, json
from pathlib import Path

from awr2944_dca import RadarProject
print(f"RadarProject class: {RadarProject}")
print(f"RadarProject module: {RadarProject.__module__}")
assert RadarProject.__module__ == 'awr2944_dca.lab', f"Unexpected module: {RadarProject.__module__}"

# Also verify the explicit import path works identically
from awr2944_dca.lab import RadarProject as RP2
assert RadarProject is RP2, "Two different RadarProject classes exist!"
print("✓ Canonical import verified – single RadarProject class")

RadarProject class: <class 'awr2944_dca.lab.RadarProject'>
RadarProject module: awr2944_dca.lab
✓ Canonical import verified – single RadarProject class


In [2]:
# Cell 2 – Verify no legacy/hardware modules loaded
forbidden = [
    'pywinauto', 'pythonnet', 'clr',
    'awr2944_dca.legacy_mmws',
    'awr2944_dca.mmws_auto',
    'serial',  # pyserial
]
loaded_forbidden = [m for m in forbidden if m in sys.modules]
assert not loaded_forbidden, f"Forbidden modules loaded: {loaded_forbidden}"
print("✓ No pywinauto, pythonnet, legacy_mmws, or serial loaded")

✓ No pywinauto, pythonnet, legacy_mmws, or serial loaded


In [3]:
# Cell 3 – Create temporary workspace
_tmpdir = tempfile.TemporaryDirectory(prefix='phase_a_smoke_')
WORKSPACE = Path(_tmpdir.name)
print(f"Workspace: {WORKSPACE}")
assert WORKSPACE.is_dir()

Workspace: C:\Users\khams008\AppData\Local\Temp\phase_a_smoke_4trpod3m


In [4]:
# Cell 4 – RadarProject.create()
proj = RadarProject.create(name='test_experiment', path=WORKSPACE)
print(f"Project root: {proj.root}")
print(f"Project name: {proj.name}")
assert proj.name == 'test_experiment'
assert proj.root == WORKSPACE / 'test_experiment'
assert proj.root.is_dir()
print("✓ RadarProject.create() succeeded")

Project root: C:\Users\khams008\AppData\Local\Temp\phase_a_smoke_4trpod3m\test_experiment
Project name: test_experiment
✓ RadarProject.create() succeeded


In [5]:
# Cell 5 – No path duplication
# Verify 'test_experiment' does NOT appear twice in the path
parts = proj.root.parts
count = parts.count('test_experiment')
assert count == 1, f"Project name appears {count} times in path: {proj.root}"
print(f"✓ No path duplication: {proj.root}")

✓ No path duplication: C:\Users\khams008\AppData\Local\Temp\phase_a_smoke_4trpod3m\test_experiment


In [6]:
# Cell 6 – Generated directory tree
root = proj.root
expected_dirs = ['profiles', 'captures', '.awr2944']
expected_files = ['awr2944.toml', 'profiles/smoke_v1.toml']

for d in expected_dirs:
    assert (root / d).is_dir(), f"Missing directory: {d}"
    print(f"  ✓ {d}/")

for f in expected_files:
    assert (root / f).is_file(), f"Missing file: {f}"
    print(f"  ✓ {f}")

print("\nFull tree:")
for p in sorted(root.rglob('*')):
    rel = p.relative_to(root)
    if p.is_dir():
        print(f"  {rel}/")
    else:
        print(f"  {rel}  ({p.stat().st_size} bytes)")
print("✓ Directory tree verified")

  ✓ profiles/
  ✓ captures/
  ✓ .awr2944/
  ✓ awr2944.toml
  ✓ profiles/smoke_v1.toml

Full tree:
  .awr2944/
  .awr2944\local.toml  (259 bytes)
  awr2944.toml  (222 bytes)
  captures/
  profiles/
  profiles\smoke_v1.toml  (55 bytes)
✓ Directory tree verified


In [7]:
# Cell 7 – awr2944.toml creation and content
import tomli

toml_path = root / 'awr2944.toml'
with open(toml_path, 'rb') as f:
    toml_data = tomli.load(f)

print(json.dumps(toml_data, indent=2))
assert toml_data['project']['name'] == 'test_experiment'
assert 'defaults' in toml_data
assert 'network' in toml_data
print("✓ awr2944.toml content verified")

{
  "project": {
    "name": "test_experiment",
    "id": "",
    "schema_version": 2
  },
  "defaults": {
    "frames": 9,
    "guard_frames": 1,
    "profile": "smoke_v1"
  },
  "network": {
    "host_ip": "192.168.33.30",
    "dca_ip": "192.168.33.180",
    "config_port": 4096,
    "data_port": 4098
  }
}
✓ awr2944.toml content verified


In [8]:
# Cell 8 – .awr2944/local.toml behavior
local_path = root / '.awr2944' / 'local.toml'
assert local_path.is_file(), "local.toml not created"

with open(local_path, 'rb') as f:
    local_data = tomli.load(f)

print(json.dumps(local_data, indent=2))
assert 'serial' in local_data
assert local_data['serial']['com_port'] == 'COM8'
assert local_data['serial']['aux_com_port'] == 'COM7'
assert 'dca_tools' in local_data
print("✓ local.toml content verified (COM8/COM7 as inert default strings)")

{
  "serial": {
    "com_port": "COM8",
    "aux_com_port": "COM7",
    "baud_rate": 115200
  },
  "dca_tools": {
    "control_exe": "C:\\ti\\mmwave_studio\\PostProc\\DCA1000EVM_CLI_Control.exe",
    "record_exe": "C:\\ti\\mmwave_studio\\PostProc\\DCA1000EVM_CLI_Record.exe",
    "cf_json": "C:\\ti\\cf.json"
  }
}
✓ local.toml content verified (COM8/COM7 as inert default strings)


In [9]:
# Cell 9 – .gitignore contents (written when git_init=True)
# Create a second project with git_init=True
proj_git = RadarProject.create(name='test_git_project', path=WORKSPACE, git_init=True)
gitignore = proj_git.root / '.gitignore'

if gitignore.is_file():
    content = gitignore.read_text()
    print(content)
    assert '.awr2944/local.toml' in content
    assert 'captures/' in content
    print("✓ .gitignore contains expected entries")
else:
    print("⚠ git not available on this system – .gitignore not created (expected in CI without git)")
    print("✓ Graceful fallback: no crash when git is absent")

git not found, skipping git initialization.


⚠ git not available on this system – .gitignore not created (expected in CI without git)
✓ Graceful fallback: no crash when git is absent


In [10]:
# Cell 10 – Portable config round-trip
proj.config.portable.frames = 16
proj.config.portable.host_ip = '10.0.0.1'
proj.config.save()

# Reload from disk via fresh ProjectConfig
from awr2944_dca._config import ProjectConfig
reloaded = ProjectConfig(proj.root)
assert reloaded.portable.frames == 16, f"Expected 16, got {reloaded.portable.frames}"
assert reloaded.portable.host_ip == '10.0.0.1', f"Expected 10.0.0.1, got {reloaded.portable.host_ip}"
assert reloaded.portable.project_name == 'test_experiment'
print(f"✓ Portable config round-trip: frames={reloaded.portable.frames}, host_ip={reloaded.portable.host_ip}")

✓ Portable config round-trip: frames=16, host_ip=10.0.0.1


In [11]:
# Cell 11 – Local config round-trip
proj.config.local.com_port = 'COM99'
proj.config.local.dca_control_exe = r'D:\TI\DCA1000EVM_CLI_Control.exe'
proj.config.save()

reloaded2 = ProjectConfig(proj.root)
assert reloaded2.local.com_port == 'COM99', f"Expected COM99, got {reloaded2.local.com_port}"
assert reloaded2.local.dca_control_exe == r'D:\TI\DCA1000EVM_CLI_Control.exe'
print(f"✓ Local config round-trip: com_port={reloaded2.local.com_port}")

# Windows Path round-trip with backslashes
assert '\\' in reloaded2.local.dca_control_exe or '/' in reloaded2.local.dca_control_exe
print(f"✓ Windows path preserved: {reloaded2.local.dca_control_exe}")

✓ Local config round-trip: com_port=COM99
✓ Windows path preserved: D:\TI\DCA1000EVM_CLI_Control.exe


In [12]:
# Cell 12 – RadarProject.open()
proj2 = RadarProject.open(proj.root)
assert proj2.root == proj.root
assert proj2.name == 'test_experiment'
assert proj2.config.portable.frames == 16  # persisted value
print(f"✓ RadarProject.open() succeeded: name={proj2.name}, frames={proj2.config.portable.frames}")

✓ RadarProject.open() succeeded: name=test_experiment, frames=16


In [13]:
# Cell 13 – RadarProject.open_here() from a deeply nested directory
deep_dir = proj.root / 'captures' / 'run_001' / 'raw' / 'subdir'
deep_dir.mkdir(parents=True, exist_ok=True)

orig_cwd = os.getcwd()
try:
    os.chdir(deep_dir)
    proj3 = RadarProject.open_here()
    assert proj3.root == proj.root
    assert proj3.name == 'test_experiment'
    print(f"✓ open_here() from {deep_dir.relative_to(proj.root)} resolved to {proj3.root}")
finally:
    os.chdir(orig_cwd)

✓ open_here() from captures\run_001\raw\subdir resolved to C:\Users\khams008\AppData\Local\Temp\phase_a_smoke_4trpod3m\test_experiment


In [14]:
# Cell 14 – Reconstruction after deleting all in-memory project objects
saved_root = proj.root
del proj, proj2, proj3, reloaded, reloaded2

# Reconstruct from disk
proj_recon = RadarProject.open(saved_root)
assert proj_recon.name == 'test_experiment'
assert proj_recon.config.portable.frames == 16
assert proj_recon.config.local.com_port == 'COM99'
print(f"✓ Reconstruction from disk: name={proj_recon.name}, frames={proj_recon.config.portable.frames}")

✓ Reconstruction from disk: name=test_experiment, frames=16


In [15]:
# Cell 15 – Old project.json fallback
legacy_dir = WORKSPACE / 'legacy_project'
legacy_dir.mkdir()

# Write a project.json like the old system used
legacy_proj = {
    'name': 'legacy_test',
    'project_id': 'abc123',
    'schema_version': 1,
}
(legacy_dir / 'project.json').write_text(json.dumps(legacy_proj))

legacy = RadarProject.open(legacy_dir)
assert legacy.name == 'legacy_test', f"Expected 'legacy_test', got '{legacy.name}'"
print(f"✓ Legacy project.json fallback: name={legacy.name}")

✓ Legacy project.json fallback: name=legacy_test


In [16]:
# Cell 16 – Reopening does not rewrite configuration unexpectedly
import hashlib

toml_before = (saved_root / 'awr2944.toml').read_bytes()
local_before = (saved_root / '.awr2944' / 'local.toml').read_bytes()

# Open the project (should NOT rewrite files)
proj_reopen = RadarProject.open(saved_root)

toml_after = (saved_root / 'awr2944.toml').read_bytes()
local_after = (saved_root / '.awr2944' / 'local.toml').read_bytes()

assert toml_before == toml_after, "awr2944.toml was modified on reopen!"
assert local_before == local_after, "local.toml was modified on reopen!"
print("✓ Reopening project does not rewrite configuration files")

✓ Reopening project does not rewrite configuration files


In [17]:
# Cell 17 – Re-creating raises clear error
try:
    RadarProject.create(name='test_experiment', path=WORKSPACE)
    assert False, "Should have raised FileExistsError"
except FileExistsError as e:
    print(f"✓ Re-creation correctly raises FileExistsError: {e}")

✓ Re-creation correctly raises FileExistsError: Cannot create project: C:\Users\khams008\AppData\Local\Temp\phase_a_smoke_4trpod3m\test_experiment already exists.


In [18]:
# Cell 18 – Project creation does not import hardware code
hardware_modules = [
    'awr2944_dca.direct_udp_capture',
    'awr2944_dca.capture_session',
    'awr2944_dca.dca_cli',
    'awr2944_dca.headless_serial',
    'awr2944_dca.sdk_cli_profile',
    'awr2944_dca.dsp',
    'awr2944_dca.dsp_pipeline',
    'serial',
    'pywinauto',
    'pythonnet',
]
loaded_hw = [m for m in hardware_modules if m in sys.modules]
assert not loaded_hw, f"Hardware modules loaded during project scaffolding: {loaded_hw}"
print("✓ No hardware modules loaded during project creation/open workflow")

✓ No hardware modules loaded during project creation/open workflow


In [19]:
# Cell 19 – Cleanup
_tmpdir.cleanup()
assert not Path(_tmpdir.name).exists(), "Temp directory not cleaned up"
print("✓ Temporary workspace cleaned up")

✓ Temporary workspace cleaned up


In [20]:
# Cell 20 – Final marker
print("PHASE_A_NOTEBOOK_PASS")

PHASE_A_NOTEBOOK_PASS
